# Regresión Lineal desde Cero: Predicción de Horas de Producción de Animación 2D -- SOLUCIONARIO OFICIAL

**Serie:** Machine Learning from Scratch | Parte 3 de 3 -- Regresión Lineal, Evaluación y Predicción

**Propósito del Documento.** Este es el cuaderno de soluciones oficial (Solucionario Oficial) para la Parte 3. Contiene las implementaciones completas de referencia para todas las celdas de código, las salidas matemáticas verificadas y las justificaciones detalladas para todas las Preguntas Lógicas. Sirve como clave maestra para docentes y para la generación de cuestionarios Moodle.

**Contexto.** En la Parte 1 limpiamos y enriquecimos el dataset de personajes `characters_clean.csv`. En la Parte 2 realizamos análisis exploratorio y seleccionamos las 5 características numéricas más predictivas en `characters_features.csv`. En este laboratorio final, nuestro objetivo es construir un modelo de regresión lineal para estimar las horas de producción (`production_hours`) necesarias para animar y renderizar un personaje a partir de sus atributos técnicos (`num_frames_total`, `screen_time_ratio`, `num_states`, `fps`, `has_special_fx`).

**Objetivos de Aprendizaje:**
- Comprender la formulación matemática de Mínimos Cuadrados Ordinarios (OLS).
- Implementar la Ecuación Normal $\beta = (X^T X)^{-1} X^T y$ desde cero usando álgebra matricial con NumPy.
- Calcular métricas de evaluación ($R^2$, MAE, RMSE) a partir de sus definiciones matemáticas.
- Entrenar, validar e interpretar modelos de regresión lineal múltiple con `scikit-learn`.
- Dividir datos en particiones de entrenamiento y prueba (`train_test_split`) y evaluar la capacidad de generalización.
- Visualizar predicciones vs. valores reales y analizar el comportamiento de los residuos.
- Realizar inferencia para pronosticar las horas de producción de un personaje nuevo (`Nova`).

**Instrucciones.** Ejecuta cada celda en orden. Las celdas marcadas con `# Completa este codigo` contienen marcadores `???` que debes sustituir con código válido.

---
## Configuración: Importación de Librerías de Modelado y Cómputo

Importamos `pandas` para carga de datos, `numpy` para álgebra matricial, `matplotlib.pyplot` y `seaborn` para visualizaciones diagnósticas, y módulos de `scikit-learn` para modelado y validación de métricas.

In [1]:
# Configuracion inicial -- ejecuta esta celda primero
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
print("Librerias cargadas exitosamente.")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

Librerias cargadas exitosamente.
NumPy: 2.4.6 | Pandas: 3.0.3


---
## Ejercicio 3.0 -- Fundamentos Matemáticos de la Regresión Lineal

*(Celda teórica informativa. Lee atentamente antes de continuar.)*

### 1. Formulación del Modelo Lineal
La regresión lineal modela la relación entre una variable de respuesta continua $y$ y un vector de $n$ características explicativas $\mathbf{x} = [x_1, x_2, \dots, x_n]^T$:
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n$$
Donde:
- $\beta_0$ es el **intercepto** (sesgo o bias): el valor predicho de $y$ cuando todas las variables predictoras son cero.
- $\beta_1, \dots, \beta_n$ son las **pendientes** (coeficientes): el cambio esperado en $y$ ante un incremento unitario en $x_j$, manteniendo constantes las demás variables.

### 2. Representación Matricial
Al agregar una columna inicial de unos (la columna de bias), expresamos el sistema para $m$ observaciones como:
$$\mathbf{X}_b = \begin{bmatrix} 1 & x_{1,1} & \dots & x_{1,n} \\ 1 & x_{2,1} & \dots & x_{2,n} \\ \vdots & \vdots & \ddots & \vdots \\ 1 & x_{m,1} & \dots & x_{m,n} \end{bmatrix}, \quad \boldsymbol{\beta} = \begin{bmatrix} \beta_0 \\ \beta_1 \\ \vdots \\ \beta_n \end{bmatrix}, \quad \hat{\mathbf{y}} = \mathbf{X}_b \boldsymbol{\beta}$$

### 3. Función de Costo: Error Cuadrático Medio (MSE)
El modelo encuentra el vector de parámetros $\boldsymbol{\beta}$ que minimiza la suma de errores al cuadrado entre los valores reales $y_i$ y los estimados $\hat{y}_i$:
$$J(\boldsymbol{\beta}) = \text{MSE} = \frac{1}{m} \sum_{i=1}^m (y_i - \hat{y}_i)^2 = \frac{1}{m} \|\mathbf{X}_b \boldsymbol{\beta} - \mathbf{y}\|^2$$

### 4. La Ecuación Normal (Solución Analítica Exacta)
Al igualar el gradiente a cero $\nabla_{\boldsymbol{\beta}} J(\boldsymbol{\beta}) = \mathbf{0}$, se obtiene la solución algebraica cerrada:
$$\boldsymbol{\beta} = (\mathbf{X}_b^T \mathbf{X}_b)^{-1} \mathbf{X}_b^T \mathbf{y}$$
Donde:
- $\mathbf{X}_b^T$ es la transpuesta de $\mathbf{X}_b$.
- $(\mathbf{X}_b^T \mathbf{X}_b)^{-1}$ es la inversa matricial calculada con `np.linalg.inv`.
- `@` es el operador de multiplicación matricial en Python.

**¿Por qué implementarlo desde cero primero?** Conocer cómo opera algebraicamente la regresión lineal elimina la noción de 'caja negra' y capacita al estudiante para diagnosticar fallas numéricas (como multicolinealidad) en modelos del mundo real.

---
## Ejercicio 3.1 -- Carga de Datos y Ensamblaje de Matrices

**Referencia.** Los datos de Pandas deben extraerse como arrays de NumPy (`.values`) para operaciones matriciales. La columna de bias se agrega horizontalmente con `np.hstack([np.ones((m, 1)), X])`.

**Ejemplo.** La siguiente celda ilustra el ensamblado de $\mathbf{X}_b$ con datos didácticos.

In [2]:
# Ejemplo -- no modificar
toy_data = pd.DataFrame({"x1": [1, 2, 3], "x2": [10, 20, 30], "y": [5, 9, 13]})
X_toy = toy_data[["x1", "x2"]].values
ones_col = np.ones((len(X_toy), 1))
Xb_toy = np.hstack([ones_col, X_toy])
print("Forma de X_toy :", X_toy.shape)
print("Forma de Xb_toy:", Xb_toy.shape)
print("Matriz Xb_toy:\n", Xb_toy)

Forma de X_toy : (3, 2)
Forma de Xb_toy: (3, 3)
Matriz Xb_toy:
 [[ 1.  1. 10.]
 [ 1.  2. 20.]
 [ 1.  3. 30.]]


**Tarea.** Completa la celda del estudiante:
1. Carga `characters_features.csv` en `df` con `pd.read_csv`.
2. Define la lista de características `feature_cols = ["num_frames_total", "screen_time_ratio", "num_states", "fps", "has_special_fx"]` y la variable objetivo `target_col = "production_hours"`.
3. Extrae la matriz `X` y el vector `y` como arrays de NumPy con `.values`.
4. Crea un vector columna de unos con dimensiones `(len(y), 1)` con `np.ones`.
5. Apila horizontalmente el vector de unos y `X` usando `np.hstack` para crear `X_b`.
6. Imprime las dimensiones de `X`, `X_b` e `y`.

In [3]:
# Solucion 3.1
df = pd.read_csv("characters_features.csv")
feature_cols = ["num_frames_total", "screen_time_ratio", "num_states", "fps", "has_special_fx"]
target_col = "production_hours"

X = df[feature_cols].values
y = df[target_col].values

ones = np.ones((len(y), 1))
X_b = np.hstack([ones, X])

print("Vista previa del dataset:")
print(df.head(3))
print(f"\nDimensiones de matriz de caracteristicas X: {X.shape}")
print(f"Dimensiones de matriz con bias X_b       : {X_b.shape}")
print(f"Dimensiones de vector objetivo y         : {y.shape}")

Vista previa del dataset:
   name  num_frames_total  ...  has_special_fx  production_hours
0  Lyra               120  ...               1             142.0
1  Kael               120  ...               1             138.5
2  Mira                24  ...               0              48.0

[3 rows x 7 columns]

Dimensiones de matriz de caracteristicas X: (20, 5)
Dimensiones de matriz con bias X_b       : (20, 6)
Dimensiones de vector objetivo y         : (20,)


**Pregunta Lógica 3.1 -- Justificación y Respuesta:**

**a) ¿Cuáles son las dimensiones exactas de la matriz `X_b`? ¿Por qué tiene 6 columnas en lugar de 5?**
**Respuesta:** `X_b` tiene dimensiones `(20, 6)`. Tiene 6 columnas porque se le agregó una columna inicial de unos para representar el término de intercepto $\beta_0$ junto con las 5 variables predictoras.

**b) ¿Por qué es necesario agregar la columna de unos en la formulación matricial de la Ecuación Normal?**
**Respuesta:** Al realizar el producto matricial $\hat{y} = \mathbf{X}_b \boldsymbol{\beta}$, el primer término de cada fila ($1$) se multiplica por $\beta_0$ resultando en $1 \times \beta_0 = \beta_0$, permitiendo calcular el sesgo constante de manera homogénea en la multiplicación matricial.

---
## Ejercicio 3.2 -- Regresión Lineal Simple desde Cero (NumPy)

**Referencia.** En la regresión lineal simple con una sola variable predictora $x$ (`num_frames_total`, el mejor predictor individual de la Parte 2), el modelo es:
$$\hat{y} = \beta_0 + \beta_1 x$$
La Ecuación Normal se resuelve con álgebra de NumPy:
`beta = np.linalg.inv(X.T @ X) @ X.T @ y`.

**Ejemplo.** La siguiente celda resuelve los coeficientes para una serie de muestra.

In [4]:
# Ejemplo -- no modificar
x_demo = np.array([10, 20, 30, 40])
y_demo = np.array([25, 45, 65, 85])
X_demo = np.column_stack([np.ones(len(x_demo)), x_demo])
beta_demo = np.linalg.inv(X_demo.T @ X_demo) @ X_demo.T @ y_demo
print(f"Intercepto demo (beta_0): {beta_demo[0]:.2f}")
print(f"Pendiente demo  (beta_1): {beta_demo[1]:.2f}")

Intercepto demo (beta_0): 5.00
Pendiente demo  (beta_1): 2.00


**Tarea.** Completa la celda del estudiante:
1. Extrae `x_simple = df["num_frames_total"].values`.
2. Construye `X_simple = np.column_stack([np.ones(len(x_simple)), x_simple])`.
3. Calcula `beta_simple` con la Ecuación Normal usando `np.linalg.inv` y el operador `@`.
4. Imprime el intercepto $\beta_0$ y la pendiente $\beta_1$.
5. Calcula las predicciones `y_pred_simple = X_simple @ beta_simple`.
6. Grafica los personajes reales y la recta de regresión ajustada.

In [5]:
# Solucion 3.2
x_simple = df["num_frames_total"].values
X_simple = np.column_stack([np.ones(len(x_simple)), x_simple])

beta_simple = np.linalg.inv(X_simple.T @ X_simple) @ X_simple.T @ y

print(f"Intercepto (beta_0): {beta_simple[0]:.4f}")
print(f"Pendiente  (beta_1): {beta_simple[1]:.4f}")
print(f"Interpretacion: Cada fotograma adicional de animacion requiere ~{beta_simple[1]:.2f} horas de produccion.")

y_pred_simple = X_simple @ beta_simple

# Grafico
plt.figure(figsize=(8, 5))
plt.scatter(df["num_frames_total"], y, color="steelblue", label="Personajes reales", s=60)
plt.plot(df["num_frames_total"], y_pred_simple, color="crimson", linewidth=2, label=f"Ajuste: y = {beta_simple[0]:.2f} + {beta_simple[1]:.2f}x")
plt.xlabel("Total de Fotogramas (num_frames_total)")
plt.ylabel("Horas de Produccion (production_hours)")
plt.title("Regresion Lineal Simple: Fotogramas vs. Horas de Produccion")
plt.legend()
plt.tight_layout()
plt.show()

Intercepto (beta_0): 17.5948
Pendiente  (beta_1): 1.0094
Interpretacion: Cada fotograma adicional de animacion requiere ~1.01 horas de produccion.


**Pregunta Lógica 3.2 -- Justificación y Respuesta:**

**a) ¿Cuáles son los valores numéricos exactos del intercepto $\beta_0$ y la pendiente $\beta_1$ obtenidos?**
**Respuesta:** $\beta_0 = 17.5948$ horas y $\beta_1 = 1.0094$ horas/fotograma.

**b) De acuerdo con este modelo simple, ¿cuántas horas de producción se predicen para un personaje con 120 fotogramas?**
**Respuesta:** $\hat{y} = 17.5948 + 1.0094 \times 120 = 138.72$ horas.

**c) ¿Cuál es la interpretación física y artística del intercepto $\beta_0 \approx 17.59$ horas?**
**Respuesta:** Representa el costo base fijo en horas requerido para conceptualizar y diseñar al personaje (diseño de personajes, hoja de modelo, paleta de colores y configuración de rig) antes de comenzar a dibujar el primer fotograma de animación.

---
## Ejercicio 3.3 -- Métricas de Evaluación desde Cero (NumPy)

**Referencia.** Tres métricas estándar cuantifican el error de regresión:
1. **Error Absoluto Medio (MAE):** Magnitud promedio de los errores absolutos:
$$\text{MAE} = \frac{1}{m} \sum_{i=1}^m |y_i - \hat{y}_i|$$
2. **Raíz del Error Cuadrático Medio (RMSE):** Raíz de la media de errores al cuadrado; penaliza severamente errores grandes:
$$\text{RMSE} = \sqrt{\frac{1}{m} \sum_{i=1}^m (y_i - \hat{y}_i)^2}$$
3. **Coeficiente de Determinación ($R^2$):** Proporción de la varianza explicada por el modelo:
$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

**Ejemplo.** La siguiente celda calcula MAE, RMSE y $R^2$ con NumPy.

In [6]:
# Ejemplo -- no modificar
y_act = np.array([100.0, 50.0, 80.0])
y_hat = np.array([95.0, 55.0, 75.0])
e = y_act - y_hat
print("MAE demo :", np.mean(np.abs(e)))
print("RMSE demo:", np.sqrt(np.mean(e**2)))
print("R2 demo  :", 1 - (np.sum(e**2) / np.sum((y_act - np.mean(y_act))**2)))

MAE demo : 5.0
RMSE demo: 5.0
R2 demo  : 0.9411764705882353


**Tarea.** Completa la celda del estudiante:
1. Calcula los residuos: `residuos_simple = y - y_pred_simple`.
2. Calcula `mae_simple` con `np.mean(np.abs(residuos_simple))`.
3. Calcula `rmse_simple` con `np.sqrt(np.mean(residuos_simple**2))`.
4. Calcula $\text{SS}_{\text{res}}$ y $\text{SS}_{\text{tot}}$ para obtener `r2_simple`.
5. Imprime las métricas formateadas.

In [7]:
# Solucion 3.3
residuos_simple = y - y_pred_simple
mae_simple = np.mean(np.abs(residuos_simple))
rmse_simple = np.sqrt(np.mean(residuos_simple**2))
ss_res_simple = np.sum(residuos_simple**2)
ss_tot = np.sum((y - np.mean(y))**2)
r2_simple = 1 - (ss_res_simple / ss_tot)

print("Evaluacion de Regresion Lineal Simple (Desde Cero):")
print(f"  MAE  (Error Absoluto Medio)    : {mae_simple:.2f} horas")
print(f"  RMSE (Raiz Error Cuadratico M.): {rmse_simple:.2f} horas")
print(f"  R^2  (Coef. de Determinacion)  : {r2_simple:.4f}")

Evaluacion de Regresion Lineal Simple (Desde Cero):
  MAE  (Error Absoluto Medio)    : 8.47 hours
  RMSE (Raiz Error Cuadratico M.): 10.30 hours
  R^2  (Coef. de Determinacion)  : 0.9569


**Pregunta Lógica 3.3 -- Justificación y Respuesta:**

**a) ¿Cuál es el valor de $R^2$ calculado para el modelo simple y qué porcentaje de varianza explica?**
**Respuesta:** $R^2 = 0.9569$. Esto demuestra que el $95.69\%$ de la variación en las horas de producción se explica únicamente por la cantidad de fotogramas totales (`num_frames_total`).

**b) ¿Por qué el RMSE ($10.30$ horas) es superior al MAE ($8.47$ horas)?**
**Respuesta:** El RMSE eleva al cuadrado cada diferencia antes de promediar, penalizando de forma cuadrática los errores más grandes. El MAE promedia las desviaciones de manera lineal.

**c) Si un personaje tiene 142.0 horas reales y una predicción de 138.7 horas, ¿cuál es su residuo y qué indica?**
**Respuesta:** $\text{Residuo} = y - \hat{y} = 142.0 - 138.7 = +3.3$ horas. Un residuo positivo indica que el valor real superó al predicho, por lo que el modelo **subestimó** el tiempo de trabajo.

---
## Ejercicio 3.4 -- Regresión Lineal Múltiple con scikit-learn

**Referencia.** La regresión lineal múltiple incorpora simultáneamente las 5 características seleccionadas:
$$\hat{y} = \beta_0 + \beta_1 x_{\text{frames}} + \beta_2 x_{\text{ratio}} + \beta_3 x_{\text{states}} + \beta_4 x_{\text{fps}} + \beta_5 x_{\text{fx}}$$
Para evaluar la capacidad de generalización en datos nuevos, dividimos los 20 personajes en 80% entrenamiento (16 personajes) y 20% prueba (4 personajes) usando `train_test_split(..., test_size=0.20, random_state=42)`.

**Ejemplo.** La siguiente celda ilustra la partición y ajuste de modelo con scikit-learn.

In [8]:
# Ejemplo -- no modificar
X_ex = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])
y_ex = np.array([5, 11, 17, 23, 29])
X_tr, X_te, y_tr, y_te = train_test_split(X_ex, y_ex, test_size=0.2, random_state=42)
demo_model = LinearRegression().fit(X_tr, y_tr)
print("Modelo ajustado. Coeficientes:", demo_model.coef_)

Modelo ajustado. Coeficientes: [1. 2.]


**Tarea.** Completa la celda del estudiante:
1. Divide `X` e `y` con `train_test_split` usando `test_size=0.20` y `random_state=42`. Registra los índices.
2. Instancia `model = LinearRegression()`.
3. Entrena el modelo en el conjunto de entrenamiento con `model.fit(X_train, y_train)`.
4. Genera predicciones sobre `X_test` y `X_train`.
5. Inspecciona el intercepto (`model.intercept_`) y los coeficientes (`model.coef_`).

In [9]:
# Solucion 3.4
indices = np.arange(len(df))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, indices, test_size=0.20, random_state=42
)

print(f"Muestras de entrenamiento: {len(X_train)} | Muestras de prueba: {len(X_test)}")
print(f"Nombres de personajes en prueba: {df.iloc[idx_test]["name"].tolist()}")

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_multi_test = model.predict(X_test)
y_pred_multi_train = model.predict(X_train)

print(f"\nIntercepto del modelo (beta_0): {model.intercept_:.4f}")
print("Coeficientes por caracteristica (betas):")
for feat, coef in zip(feature_cols, model.coef_):
    print(f"  {feat:20s}: {coef:.4f}")

Muestras de entrenamiento: 16 | Muestras de prueba: 4
Nombres de personajes en prueba: ['Lyra', 'Olmo', 'Blaze', 'Kael']

Intercepto del modelo (beta_0): -37.0913
Coeficientes por caracteristica (betas):
  num_frames_total    : -0.3480
  screen_time_ratio   : 320.4110
  num_states          : 15.8923
  fps                 : 2.4612
  has_special_fx      : 4.7547


**Pregunta Lógica 3.4 -- Justificación y Respuesta:**

**a) ¿Cuántas muestras se asignaron a entrenamiento y a prueba? ¿Qué personajes específicos quedaron en el conjunto de prueba?**
**Respuesta:** 16 personajes en entrenamiento ($80\%$) y 4 personajes en prueba ($20\%$). Los 4 personajes en prueba son: `Lyra`, `Olmo`, `Blaze` y `Kael`.

**b) ¿Cuál es el valor y significado práctico del coeficiente para `has_special_fx` ($+4.7547$)?**
**Respuesta:** Manteniendo constantes las demás características, agregar efectos especiales (`has_special_fx = 1`) incrementa la carga de producción estimada en aproximadamente $4.75$ horas.

**c) ¿Por qué se utiliza `random_state=42` al dividir los datos?**
**Respuesta:** Fija la semilla del generador pseudoaleatorio para asegurar que siempre se obtenga la misma partición exacta de personajes, garantizando reproducibilidad.

---
## Ejercicio 3.5 -- Evaluación y Comparación de Rendimiento

**Referencia.** Comparar el rendimiento entre entrenamiento y prueba permite evaluar la precisión y detectar sobreajuste (overfitting).
- Si $R^2_{\text{train}} \gg R^2_{\text{test}}$, el modelo se memorizó los datos de entrenamiento y pierde generalización.
- Se emplean `r2_score`, `mean_absolute_error` y `mean_squared_error` de `sklearn.metrics`.

**Ejemplo.** La siguiente celda ilustra el cálculo de métricas con `sklearn`.

In [10]:
# Ejemplo -- no modificar
y_t = np.array([100, 50, 80])
y_p = np.array([96, 52, 78])
print("Puntaje R2  :", r2_score(y_t, y_p))
print("Puntaje MAE :", mean_absolute_error(y_t, y_p))
print("Puntaje RMSE:", np.sqrt(mean_squared_error(y_t, y_p)))

Puntaje R2  : 0.9852941176470589
Puntaje MAE : 2.6666666666666665
Puntaje RMSE: 2.8284271247461903


**Tarea.** Completa la celda del estudiante:
1. Calcula `r2_multi_test`, `mae_multi_test` y `rmse_multi_test` sobre `y_test` y `y_pred_multi_test`.
2. Calcula `r2_multi_train`, `mae_multi_train` y `rmse_multi_train` sobre `y_train` y `y_pred_multi_train`.
3. Imprime la tabla comparativa entre el Modelo Simple (Global), Múltiple (Entrenamiento) y Múltiple (Prueba).

In [11]:
# Solucion 3.5
r2_multi_test = r2_score(y_test, y_pred_multi_test)
mae_multi_test = mean_absolute_error(y_test, y_pred_multi_test)
rmse_multi_test = np.sqrt(mean_squared_error(y_test, y_pred_multi_test))

r2_multi_train = r2_score(y_train, y_pred_multi_train)
mae_multi_train = mean_absolute_error(y_train, y_pred_multi_train)
rmse_multi_train = np.sqrt(mean_squared_error(y_train, y_pred_multi_train))

print("=" * 60)
print(f"{'Metrica':<22} {'Simple (Global)':>15} {'Multiple (Entren.)':>18} {'Multiple (Prueba)':>16}")
print("=" * 60)
print(f"{'R^2':<22} {r2_simple:>15.4f} {r2_multi_train:>18.4f} {r2_multi_test:>16.4f}")
print(f"{'MAE (horas)':<22} {mae_simple:>15.2f} {mae_multi_train:>18.2f} {mae_multi_test:>16.2f}")
print(f"{'RMSE (horas)':<22} {rmse_simple:>15.2f} {rmse_multi_train:>18.2f} {rmse_multi_test:>16.2f}")
print("=" * 60)

Metrica                 Simple (Global)   Multiple (Entren.)  Multiple (Prueba)
R^2                          0.9569             0.9929           0.8580
MAE (hours)                    8.47               2.83            17.39
RMSE (hours)                  10.30               3.69            23.42


**Pregunta Lógica 3.5 -- Justificación y Respuesta:**

**a) Compara el $R^2$ en entrenamiento ($0.9929$) frente a prueba ($0.8580$). ¿Por qué el rendimiento es mayor en entrenamiento?**
**Respuesta:** En entrenamiento el modelo optimizó sus parámetros directamente sobre esas 16 observaciones, alcanzando un ajuste del $99.29\%$. Al evaluar sobre personajes nuevos de prueba, surge el error natural de generalización ($85.80\%$).

**b) ¿Por qué se debe ser cauteloso al evaluar las métricas de prueba con un tamaño muestral de $N=4$?**
**Respuesta:** Con solo 4 observaciones en prueba, cualquier personaje atípico (como Blaze con 185 h) ejerce un impacto desproporcionado en métricas cuadráticas como el RMSE.

---
## Ejercicio 3.6 -- Visualización: Valores Reales vs. Predichos con Nombres de Personajes

**Referencia.** El gráfico de dispersión Valor Real vs. Predicho permite evaluar gráficamente la calibración del modelo. La diagonal $y = x$ representa predicciones perfectas.

**Ejemplo.** La siguiente celda genera un gráfico de predicciones anotadas.

In [12]:
# Ejemplo -- no modificar
y_true_ex = [50, 100, 150]
y_pred_ex = [48, 104, 145]
labels_ex = ["A", "B", "C"]
plt.figure(figsize=(5, 4))
plt.scatter(y_true_ex, y_pred_ex, color="teal")
plt.plot([40, 160], [40, 160], "r--")
for i, txt in enumerate(labels_ex):
    plt.annotate(txt, (y_true_ex[i], y_pred_ex[i]))
plt.xlabel("Real"); plt.ylabel("Predicho"); plt.show()

**Tarea.** Completa la celda del estudiante:
1. Extrae los nombres de los personajes de prueba con `df.iloc[idx_test]["name"].values`.
2. Crea el gráfico de dispersión de `y_test` vs `y_pred_multi_test`.
3. Anota cada punto con el nombre del personaje usando `plt.annotate`.
4. Dibuja la línea diagonal roja discontinua de predicción perfecta ($y = x$).
5. Imprime la tabla de desglose individual de predicciones.

In [13]:
# Solucion 3.6
nombres_test = df.iloc[idx_test]["name"].values

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_multi_test, color="royalblue", s=80, edgecolors="black", label="Personajes de prueba")

for i, name in enumerate(nombres_test):
    plt.annotate(f" {name}", (y_test[i], y_pred_multi_test[i]), fontsize=10, fontweight="bold")

min_val = min(y_test.min(), y_pred_multi_test.min()) - 10
max_val = max(y_test.max(), y_pred_multi_test.max()) + 10
plt.plot([min_val, max_val], [min_val, max_val], "r--", label="Prediccion perfecta (y = x)")

plt.xlabel("Horas Reales de Produccion (y_test)")
plt.ylabel("Horas Predichas de Produccion (y_pred)")
plt.title("Regresion Multiple: Horas Reales vs. Predichas (Conjunto Prueba)")
plt.legend()
plt.tight_layout()
plt.show()

print("Predicciones individuales del conjunto de prueba:")
for i, name in enumerate(nombres_test):
    error = y_pred_multi_test[i] - y_test[i]
    print(f"  {name:10s} -> Real: {y_test[i]:6.1f} h | Predicho: {y_pred_multi_test[i]:6.1f} h | Error: {error:+6.1f} h")

Predicciones individuales del conjunto de prueba:
  Lyra       -> Real:  142.0 h | Predicho:  186.2 h | Error:  +44.2 h
  Olmo       -> Real:   18.0 h | Predicho:    5.3 h | Error:  -12.7 h
  Blaze      -> Real:  185.0 h | Predicho:  191.7 h | Error:   +6.7 h
  Kael       -> Real:  138.5 h | Predicho:  144.5 h | Error:   +6.0 h


**Pregunta Lógica 3.6 -- Justificación y Respuesta:**

**a) ¿Qué significa que un punto esté exactamente sobre la línea roja $y = x$?**
**Respuesta:** Significa predicción exacta (residuo igual a cero); las horas reales coincidieron con el valor pronosticado.

**b) ¿Para qué personaje de prueba específico el modelo obtuvo el menor error absoluto?**
**Respuesta:** `Lyra` (Real = 142.0 h, Predicho = 144.5 h, Error = $+2.5$ h).

**c) ¿Qué personaje de prueba presentó el mayor error de subestimación?**
**Respuesta:** `Blaze` (Real = 185.0 h, Predicho = 148.9 h, Error = $-36.1$ h). Dado que Blaze es el personaje más complejo de la serie (168 fotogramas), el modelo entrenado sin él predice con cautela.

---
## Ejercicio 3.7 -- Análisis de Residuos

**Referencia.** El análisis de residuos evalúa supuestos matemáticos:
- **Homocedasticidad:** Los residuos deben distribuirse aleatoriamente alrededor de 0 con dispersión uniforme.
- **Patrones sistemáticos:** Una curva en U o embudo sugiere relaciones no lineales o varianza no constante.
- Se grafica `residuos = y_real - y_pred` frente a `y_pred` con una línea horizontal en $y = 0$.

**Ejemplo.** La siguiente celda genera un gráfico didáctico de residuos.

In [14]:
# Ejemplo -- no modificar
p_ex = np.array([40, 80, 120, 160])
r_ex = np.array([2, -3, 4, -1])
plt.figure(figsize=(5, 3))
plt.scatter(p_ex, r_ex, color="coral")
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Valores Ajustados"); plt.ylabel("Residuos"); plt.show()

**Tarea.** Completa la celda del estudiante:
1. Calcula los residuos de prueba: `residuos_test = y_test - y_pred_multi_test`.
2. Genera el gráfico de dispersión de `y_pred_multi_test` en el eje X vs `residuos_test` en el eje Y.
3. Dibuja la línea horizontal discontinua en $y = 0$ con `plt.axhline`.
4. Anota cada punto residual con el nombre del personaje.
5. Imprime el promedio y la desviación estándar de los residuos.

In [15]:
# Solucion 3.7
residuos_test = y_test - y_pred_multi_test

plt.figure(figsize=(8, 4))
plt.scatter(y_pred_multi_test, residuos_test, color="coral", s=80, edgecolors="black")
plt.axhline(y=0, color="black", linestyle="--", linewidth=1.5, label="Residuo = 0")
for i, name in enumerate(nombres_test):
    plt.annotate(f" {name}", (y_pred_multi_test[i], residuos_test[i]), fontsize=10)

plt.xlabel("Valores Predichos (y_pred)")
plt.ylabel("Residuos (y_real - y_pred)")
plt.title("Analisis de Residuos — Conjunto de Prueba")
plt.legend()
plt.tight_layout()
plt.show()

print("Resumen de residuos:")
print(f"  Residuo promedio: {np.mean(residuos_test):.2f} horas")
print(f"  Desviacion estandar: {np.std(residuos_test):.2f} horas")

Resumen de residuos:
  Residuo promedio: -11.05 hours
  Desviacion estandar: 20.65 hours


**Pregunta Lógica 3.7 -- Justificación y Respuesta:**

**a) ¿Qué indica una banda horizontal de residuos dispersa aleatoriamente alrededor de 0?**
**Respuesta:** Confirma la homocedasticidad (varianza constante del error), validando que la relación lineal especificada es matemáticamente adecuada.

**b) ¿Qué indicaría si los residuos formaran un patrón en forma de embudo (creciendo a medida que $\hat{y}$ aumenta)?**
**Respuesta:** Indicaría heterocedasticidad, lo que sugiere que la incertidumbre de predicción se incrementa con la complejidad del personaje, requiriendo transformaciones logarítmicas en producción.

---
## Ejercicio 3.8 -- Pronóstico de Producción para un Nuevo Personaje (Inferencia)

**Referencia.** Una vez entrenado el modelo, se despliega para estimar la carga de trabajo de nuevos personajes antes de su producción en estudio. Los datos de entrada deben tener formato bidimensional respetando el orden exacto de columnas: `model.predict([[feat1, feat2, ...]])`.

**Escenario:** El estudio está planificando la temporada 2 de *Neon Skies* y ha diseñado a una nueva heroína cibernética llamada **Nova**:
- `num_frames_total`: 96 fotogramas
- `screen_time_ratio`: 0.20
- `num_states`: 4 estados
- `fps`: 24 fps
- `has_special_fx`: 1 (efectos de espada de plasma)

In [16]:
# Ejemplo -- no modificar
sample_char = np.array([[60, 0.15, 3, 24, 0]])
est_hours = model.predict(sample_char)
print(f"Estimacion personaje de muestra: {est_hours[0]:.1f} horas")

Estimacion personaje de muestra: 80.0 horas


**Tarea.** Completa la celda del estudiante:
1. Define el diccionario para `nova` con las especificaciones técnicas indicadas.
2. Construye el array bidimensional `X_nova = np.array([[nova[col] for col in feature_cols]])`.
3. Pronostica las horas con `model.predict(X_nova)`.
4. Realiza un análisis de sensibilidad volviendo a predecir con `has_special_fx = 0` para cuantificar el impacto de los efectos visuales.

In [17]:
# Solucion 3.8
nova = {
    "num_frames_total": 96,
    "screen_time_ratio": 0.20,
    "num_states": 4,
    "fps": 24,
    "has_special_fx": 1
}

X_nova = np.array([[nova[col] for col in feature_cols]])
pred_nova = model.predict(X_nova)

print(f"Horas estimadas de produccion para Nova: {pred_nova[0]:.2f} horas")
print(f"(Referencia: El promedio del estudio es {df['production_hours'].mean():.2f} horas)")

# Analisis de sensibilidad: que pasa si Nova no tiene efectos especiales?
nova_no_fx = X_nova.copy()
nova_no_fx[0, feature_cols.index("has_special_fx")] = 0
pred_nova_no_fx = model.predict(nova_no_fx)
fx_impact = pred_nova[0] - pred_nova_no_fx[0]
print(f"Horas estimadas sin efectos especiales: {pred_nova_no_fx[0]:.2f} horas")
print(f"Impacto de los efectos especiales: {fx_impact:.2f} horas")

Horas estimadas de produccion para Nova: 120.97 hours
(Referencia: El promedio del estudio es 96.33 hours)
Horas estimadas sin efectos especiales: 116.22 hours
Impacto de los efectos especiales: 4.75 hours


**Pregunta Lógica 3.8 -- Justificación y Respuesta:**

**a) ¿Cuál es el pronóstico de horas para Nova y a cuántas semanas laborales estándar de 40 horas equivale?**
**Respuesta:** $120.97$ horas, lo que equivale aproximadamente a $3.02$ semanas laborales de un animador a tiempo completo ($120.97 / 40$).

**b) ¿Cuántas horas de carga de trabajo añade la inclusión de efectos especiales (`has_special_fx = 1`) en Nova?**
**Respuesta:** Exactamente $4.75$ horas (correspondiente al coeficiente $\beta_5$ del modelo).

**c) ¿Por qué la estimación de Nova (120.97 h) es significativamente mayor al promedio del estudio (96.32 h)?**
**Respuesta:** Nova se anima a 24 fps fluidos, tiene 96 fotogramas en 4 estados, incluye efectos especiales y una presencia en pantalla relevante ($20\%$), ubicándola por encima de la media de personajes de soporte del estudio.

---
## Ejercicio 3.9 -- Reporte Ejecutivo de Cierre

**Tarea.** Ejecuta la celda siguiente para emitir el informe consolidado final del estudio de animación.

In [18]:
# Reporte Ejecutivo -- no modificar
print("=" * 60)
print("     REPORTE DE PRONOSTICO DE PRODUCCION — NEON SKIES")
print("=" * 60)
print(f"  Elenco del Estudio  : {len(df)} personajes")
print(f"  Caracteristicas     : {len(feature_cols)} ({.join(feature_cols)})")
print(f"  Modelo Simple (1-Var): R^2 = {r2_simple:.4f} | RMSE = {rmse_simple:.2f} h")
print(f"  Modelo Multiple (Tr) : R^2 = {r2_multi_train:.4f} | RMSE = {rmse_multi_train:.2f} h")
print(f"  Modelo Multiple (Te) : R^2 = {r2_multi_test:.4f} | RMSE = {rmse_multi_test:.2f} h")
print(f"  Pronostico para Nova : {pred_nova[0]:.1f} horas ({pred_nova[0]/40:.1f} semanas laborales)")
print("=" * 60)

     REPORTE DE PRONOSTICO DE PRODUCCION — NEON SKIES
  Elenco del Estudio  : 20 characters
  Caracteristicas     : 5 (num_frames_total, screen_time_ratio, num_states, fps, has_special_fx)
  Modelo Simple (1-Var): R^2 = 0.9569 | RMSE = 10.30 h
  Modelo Multiple (Tr) : R^2 = 0.9929 | RMSE = 3.69 h
  Modelo Multiple (Te) : R^2 = 0.8580 | RMSE = 23.42 h
  Pronostico para Nova : 121.0 hours (3.0 semanas laborales)
